# 📈 Backtest du Portefeuille K du projet *Aim for the Moon*

Ce notebook présente l’analyse complète du portefeuille *Aim for the Moon*, un portefeuille multi‑ETF construit autour de plusieurs enveloppes (PEA, Assurance‑Vie, CTO) et réparti selon des poids fixes définis par l’utilisateur.

L’objectif est de :

- charger les données historiques des ETF,
- appliquer les poids du portefeuille,
- calculer les métriques de performance (annual return, volatilité, Sharpe, drawdown, VaR, ES),
- visualiser la performance cumulée, les drawdowns, les corrélations, les distributions de rendements,
- analyser la robustesse du portefeuille sur la période disponible.

Les données utilisées sont des **daily returns**, et le backtest commence automatiquement à la date où **tous les ETF ont des données valides**.


In [ ]:
# === Imports généraux ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# === Modules du projet ===
from src.data_loader import load_data
from src.portfolio import portfolio_metrics, portfolio_return
from src.static_backtest import static_backtest
from src.visualization import (
    plot_performance,
    plot_drawdown,
    plot_return_distribution,
    plot_correlation_heatmap,
    plot_correlation_matrix,
    plot_weights,
    plot_annual_returns,
    plot_rolling_volatility,
    plot_rolling_sharpe,
    plot_efficient_frontier
)
from src.optimization import optimize_portfolio, optimize_portfolio_with_constraints
# === Configuration pandas ===
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)


In [ ]:
# === Définition des tickers Yahoo Finance ===
# Chaque ligne correspond à un sous-segment du portefeuille K

tickers = {
    # --- Actions (43%) ---
    "World_large_caps": "CW8.PA",          # MSCI World
    "Small_Caps_Europe": "SMC.PA",         # Small Caps Europe
    "US_Growth": "IUSG",                   # iShares US Growth ETF
    "US_Value": "IUSV",                    # iShares US Value ETF
    "US_Mid_Caps": "IJH",                  # iShares US Mid Cap ETF
    "Small_Caps_World": "WSML.L",          # MSCI World Small Cap
    "India": "INDA",                       # iShares MSCI India ETF
    "Asia_Emerging": "EEMA",               # iShares MSCI Emerging Asia
    "Emerging_ex_Asia": "EMXC",            # iShares Emerging Markets ex-Asia

    # --- Obligations (20%) ---
    "UST_20Y+": "TLT",                     # US Treasuries 20+ ans
    "Euro_Gov_20Y+": "IBGL.L",             # Euro Government 20+ ans
    "UST_7_10Y": "IEF",                    # US Treasuries 7–10 ans
    "Euro_Gov_7_10Y": "IEGA.L",            # Euro Government 7–10 ans

    # --- Crypto (10%) ---
    "Bitcoin": "BTC-USD",
    "Ethereum": "ETH-USD",
    "Altcoins": "SOL-USD",                 # Exemple : Solana (proxy altcoins)

    # --- Immobilier (10%) ---
    "REITs_Global": "REET",                # Global REITs
    "RealEstate_Europe": "IWDP.L",         # European Property ETF
    "RealEstate_Private": "BNP.PA",        # Proxy SCPI/OPCI (non coté)

    # --- Métaux précieux (7.5%) ---
    "Gold": "GLD",                         # SPDR Gold Trust
    "Silver": "SLV",                       # iShares Silver Trust

    # --- Matières premières (7.5%) ---
    "Broad_Commodities": "PDBC",           # Bloomberg Commodity Index ETF
    "Energy": "XLE",                       # Energy Select Sector SPDR
    "Industrial_Metals": "DBB",            # Invesco Base Metals ETF

    # --- Cash (2%) ---
    "Cash": "EURUSD=X"                     # Proxy liquidités (EUR/USD)
}

# === Poids du portefeuille (somme = 100%) ===
weights = np.array([
    # Actions
    0.15, 0.05, 0.10, 0.05, 0.03, 0.02, 0.01, 0.01, 0.01,
    # Obligations
    0.05, 0.05, 0.05, 0.05,
    # Crypto
    0.05, 0.03, 0.02,
    # Immobilier
    0.06, 0.02, 0.02,
    # Métaux précieux
    0.04, 0.035,
    # Matières premières
    0.02, 0.035, 0.02,
    # Cash
    0.02
])


In [ ]:
# === Chargement des données depuis Yahoo Finance ===
# On utilise le loader robuste qui gère automatiquement la date de départ
# en fonction de la disponibilité des ETF.

data = load_data(tickers)

# Aperçu des données brutes
data.tail(10)


In [ ]:
# === Calcul des rendements journaliers ===
# On calcule les daily returns à partir des prix de clôture.
# On supprime les valeurs manquantes pour éviter les biais.

returns = data.pct_change().dropna()

# Aperçu des rendements
returns.head()


In [ ]:
# === Backtest statique ===
# On applique les poids fixes du portefeuille sur toute la période disponible.
# Le backtest calcule les rendements cumulés et toutes les métriques principales.

result = static_backtest(returns, weights)

# Aperçu des rendements cumulés
result["cumulative_returns"].tail()


In [ ]:
# === Affichage des métriques du portefeuille ===
# On affiche toutes les métriques calculées : rendement annualisé, volatilité,
# Sharpe ratio, drawdown, VaR, Expected Shortfall, meilleures/pires années, etc.

pd.DataFrame(result["metrics"], index=["Portefeuille K"]).T


In [ ]:
# === Visualisations ===
plot_performance(result["cumulative_returns"])
plot_drawdown(result["cumulative_returns"])
plot_return_distribution(result["daily_returns"])
plot_correlation_heatmap(returns)
plot_weights(weights, returns.columns)
plot_annual_returns(result["daily_returns"])
plot_rolling_volatility(result["daily_returns"])
plot_rolling_sharpe(result["daily_returns"])


In [ ]:
# === Cumulative returns par ETF + portefeuille ===

# Cumul par ETF
cum_by_etf = (1 + returns).cumprod()

# Cumul du portefeuille (déjà calculé dans result)
cum_portfolio = result["cumulative_returns"]

# On ajoute le portefeuille comme colonne à part
cum_all = cum_by_etf.copy()
cum_all["Portefeuille_K"] = cum_portfolio

# Palette de couleurs cohérente (compatible toutes versions)
colors = plt.get_cmap("tab20")(np.linspace(0, 1, len(cum_all.columns)))

# Plot
plt.figure(figsize=(14, 7))
for i, col in enumerate(cum_all.columns):
    if col == "Portefeuille_K":
        plt.plot(
            cum_all.index,
            cum_all[col],
            label=col,
            linewidth=2.5,
            color="black"
        )
    else:
        plt.plot(
            cum_all.index,
            cum_all[col],
            label=col,
            color=colors[i],
            alpha=0.8
        )

# Échelle logarithmique pour corriger le problème d’échelle
plt.yscale("log")

plt.title("Cumulative Returns (log scale) - ETFs vs Portefeuille K")
plt.xlabel("Time")
plt.ylabel("Growth (log scale)")
plt.legend(loc="upper left", ncol=2)
plt.grid(True, which="both", linestyle="--", alpha=0.5)
plt.show()


In [ ]:
import yfinance as yf

# === Benchmarks ===
benchmark_tickers = {
    "SP500": "^GSPC",    # S&P 500
    "CAC40": "^FCHI"     # CAC 40
}

bench_data = load_data(benchmark_tickers)


# Daily returns des benchmarks
bench_returns = bench_data.pct_change().dropna()


In [ ]:
# === Metrics pour portefeuille et benchmarks ===

def metrics_from_returns(ret_series, name):
    ann_return = ret_series.mean() * 252
    ann_vol = ret_series.std() * np.sqrt(252)
    sharpe = ann_return / ann_vol if ann_vol != 0 else np.nan
    cumulative = (1 + ret_series).cumprod()
    rolling_max = cumulative.cummax()
    drawdowns = (cumulative - rolling_max) / rolling_max
    max_dd = drawdowns.min()
    return {
        "name": name,
        "annual_return": ann_return,
        "annual_volatility": ann_vol,
        "sharpe_ratio": sharpe,
        "max_drawdown": max_dd
    }

metrics_list = []

# Portefeuille
metrics_list.append(metrics_from_returns(result["daily_returns"], "Portefeuille K"))

# S&P 500
metrics_list.append(metrics_from_returns(bench_returns["SP500"], "S&P 500"))

# CAC 40
metrics_list.append(metrics_from_returns(bench_returns["CAC40"], "CAC 40"))

comparison_df = pd.DataFrame(metrics_list).set_index("name")
comparison_df


In [ ]:
# === Cumulative returns comparés ===

cum_port = (1 + result["daily_returns"]).cumprod()
cum_sp = (1 + bench_returns["SP500"]).cumprod()
cum_cac = (1 + bench_returns["CAC40"]).cumprod()

plt.figure(figsize=(12, 6))
plt.plot(cum_port.index, cum_port, label="Portefeuille K", linewidth=2.5, color="black")
plt.plot(cum_sp.index, cum_sp, label="S&P 500", linewidth=1.8, color="blue")
plt.plot(cum_cac.index, cum_cac, label="CAC 40", linewidth=1.8, color="orange")

plt.title("Cumulative Returns - Portefeuille K vs Marché")
plt.xlabel("Time")
plt.ylabel("Growth (base 1)")
plt.legend()
plt.grid(True)
plt.show()


Portefeuil OPTIMISE

In [ ]:
opt = maximize_sharpe(returns)
opt_weights = opt["weights"]
